

# Обнаружение аномалий : Анализ данных банковского сектора (LOF + Isolation Forest)
**Цель ：**  

1) Загрузить и подготовить данные banks.txt (финансовые показатели).
2) Выполнить поиск аномалий алгоритмом LOF (Local Outlier Factor) в обычном режиме (novelty=False).
3) Реализовать режим обнаружения новизны (novelty=True), обучив модель на "нормальных" данных и проверив ранее найденные аномалии.
4) Провести сравнительный анализ с использованием алгоритма Isolation Forest (Изолирующий лес).
5) Изучить метрики и интерпретировать результаты, выявив наиболее критические финансовые аномалии (например, банки с отрицательным капиталом).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

# === Шаг 1. Загрузка данных ===
# Используем кодировку cp1251 для корректного чтения кириллицы из файла banks.txt
path = "banks.txt"
df = pd.read_csv(path, encoding="cp1251")

print("--- Первые 5 строк исходных данных ---")
print(df.head())

--- Первые 5 строк исходных данных ---
               Bank  Assents  OwnCapital  IndFunds  NBSLoans  IndLoans
0        «Авангард»   122109       20440     35443     32728      3319
1           «Аверс»   110741       24410     34918     13613      4924
2           «Агора»     1114         356       274       351       206
3  «Агропромкредит»    18774        2332     12047      6484       903
4         «Агророс»     7917        1157      3564      1909       492


In [2]:
# === Шаг 2. Подготовка признаков (Масштабирование) ===
# Выбираем только числовые колонки для анализа (исключаем название банка)
X = df.select_dtypes(include=[np.number]).copy()

# Стандартизация данных (Z-масштабирование)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("\n--- Данные после масштабирования (первые 5 строк) ---")
print(X_scaled.head())


--- Данные после масштабирования (первые 5 строк) ---
    Assents  OwnCapital  IndFunds  NBSLoans  IndLoans
0 -0.071960   -0.036578 -0.065488 -0.097648 -0.107472
1 -0.077810   -0.022560 -0.066141 -0.112365 -0.103954
2 -0.134225   -0.107495 -0.109261 -0.122575 -0.114295
3 -0.125137   -0.100518 -0.094608 -0.117853 -0.112768
4 -0.130724   -0.104667 -0.105166 -0.121375 -0.113669


In [3]:
# === Шаг 3. Алгоритм LOF в обычном режиме (Novelty = False) ===
#  Поиск аномалий в текущей выборке с параметрами по умолчанию
lof = LocalOutlierFactor(n_neighbors=20, novelty=False)
y_pred_lof = lof.fit_predict(X_scaled)

# Подсчет количества аномалий (-1 означает выброс)
n_anomalies = (y_pred_lof == -1).sum()
print(f"\nКоличество найденных аномалий (LOF): {n_anomalies}")

# Сохраняем результаты в основной датафрейм
df['Is_Anomaly_LOF'] = y_pred_lof


Количество найденных аномалий (LOF): 77


In [4]:
# === Шаг 4. LOF в режиме обнаружения новизны (Novelty = True) ===
#  Обучение модели на "чистых" данных и проверка выбросов

# Фильтруем данные: разделяем на нормальные (inliers) и аномальные (outliers)
X_inliers = X_scaled[y_pred_lof == 1]
X_outliers = X_scaled[y_pred_lof == -1]

# Обучаем новую модель LOF только на нормальных данных
lof_novelty = LocalOutlierFactor(n_neighbors=20, novelty=True)
lof_novelty.fit(X_inliers)

# Прогоняем ранее найденные аномалии через новую модель
y_pred_test = lof_novelty.predict(X_outliers)
n_matched = (y_pred_test == -1).sum()

print(f"\nРезультаты Novelty Detection:")
print(f"Количество совпадений: {n_matched} из {n_anomalies}")
print(f"Точность совпадения: {n_matched / n_anomalies:.2%}")


Результаты Novelty Detection:
Количество совпадений: 77 из 77
Точность совпадения: 100.00%


d:\Program Files\anaconda3\envs\ind-env\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but LocalOutlierFactor was fitted with feature names
  warnings.warn(


In [5]:
# === Шаг 5. Метод Isolation Forest ===
# Решение задачи методом изолирующего леса
# Параметр contamination устанавливается автоматически на основе результатов LOF
contamination_rate = n_anomalies / len(df)

iso_forest = IsolationForest(contamination=contamination_rate, random_state=42)
y_pred_iso = iso_forest.fit_predict(X_scaled)

df['Is_Anomaly_ISO'] = y_pred_iso
n_anomalies_iso = (y_pred_iso == -1).sum()

print(f"\nКоличество аномалий (Isolation Forest): {n_anomalies_iso}")
print(f"Используемый параметр contamination: {contamination_rate:.4f}")


Количество аномалий (Isolation Forest): 77
Используемый параметр contamination: 0.2075


In [6]:
# === Шаг 6. Анализ метрик и итоговый вывод ===
# Изучение показателей качества и списков банков

# Добавляем оценку аномальности (чем ниже значение, тем более аномален объект)
df['LOF_Score'] = lof.negative_outlier_factor_

# Поиск пересечений: банки, которые оба метода признали аномальными
common_anomalies = df[(df['Is_Anomaly_LOF'] == -1) & (df['Is_Anomaly_ISO'] == -1)]
print(f"\nКоличество общих аномалий (LOF + ISO): {len(common_anomalies)}")

# Вывод списка 10 самых аномальных банков по версии LOF
print("\n--- ТОП-10 наиболее аномальных банков (по LOF Score) ---")
top_anomalies = df.sort_values(by='LOF_Score').head(10)
print(top_anomalies[['Bank', 'Assents', 'OwnCapital', 'LOF_Score']])


Количество общих аномалий (LOF + ISO): 41

--- ТОП-10 наиболее аномальных банков (по LOF Score) ---
                             Bank   Assents  OwnCapital    LOF_Score
100                       «Траст»    473754    -1357080 -4462.044285
298               Сбербанк России  32421026     4873465   -50.111375
228                    Мособлбанк    433292     -133031   -49.131503
158                           ВТБ  15813216     1662670   -21.771059
134                Балтинвестбанк     46446      -20345   -18.121967
161                   Газпромбанк   7613174      746441    -8.582142
129                    Альфа-банк   4229025      612922    -5.881815
285                Россельхозбанк   3539546      507453    -5.026300
300  Севастопольский морской банк      3144        -784    -4.968058
164                       Генбанк     45850       -5918    -3.958041


Итоговые выводы по анализу 

Вывод 1: Сравнительный анализ алгоритмов 
LOF (Local Outlier Factor) нашел 77 аномалий, основываясь на локальной плотности. Он ищет банки, показатели которых сильно отличаются от ближайших "соседей".
Isolation Forest также выделил 77 аномалий, основываясь на структуре дерева (насколько легко "изолировать" объект).
Пересечение (41 банк): То, что 41 банк был признан аномальным обоими методами, говорит о наличии сильных аномалий. Эти объекты являются наиболее подозрительными с точки зрения разных математических подходов.

Вывод 2: Типология найденных аномалий 
Изучив таблицу ТОП-10, мы можем разделить аномалии на две группы:
"Гиганты" (Глобальные аномалии): Такие банки, как Сбербанк России и ВТБ. Их показатели активов (Assents) в сотни раз превышают средние значения. Они аномальны просто из-за своего огромного масштаба.

"Финансово нездоровые" (Технические аномалии):
Банк «Траст»: Имеет критически низкий (отрицательный) собственный капитал (OwnCapital = -1,357,080) и самый экстремальный показатель LOF_Score (-4462.04). Это явный признак финансового бедствия.
Мособлбанк, Балтинвестбанк, Генбанк: Также имеют отрицательный или аномально низкий капитал по сравнению с объемом активов.

Вывод 3: Эффективность Novelty Detection 
Показатель совпадения 100% (77 из 77) в режиме Novelty Detection подтверждает, что модель успешно "выучила" характеристики нормальных банков и безошибочно определяет ранее отфильтрованные аномалии как "новые/незнакомые" объекты.
